# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining
## SCADA-Core Automática — Grupo 04: Classificação e Seleção de Grãos por Visão Computacional

Neste notebook implementamos algoritmos completos de **Forward Chaining** (Data-Driven com Resolução de Conflitos e Ponto Fixo de van Emden-Kowalski) e **Backward Chaining** (Goal-Driven com Resolução SLD e detecção de ciclos em grafo AND-OR) para diagnóstico automatizado em tempo de execução na planta industrial de seleção de grãos.

O sistema conta também com módulo de rastreabilidade e explicabilidade (**Audit Trail / Explanation Facility**) para responder formalmente às perguntas operacionais do SCADA (*"HOW"* e *"WHY"*).


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionários em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from enum import Enum
from typing import Set, Tuple, List, Dict, Optional, Any
import time

class Severidade(Enum):
    CRITICA = 1      # Trip de segurança / Parada imediata
    ALTA = 2         # Falha de subsistema / Ejeção inibida
    MEDIA = 3        # Degradação de processo / Alerta operacional
    BAIXA = 4        # Informativo / Manutenção preventiva

@dataclass
class Rule:
    rule_id: str
    antecedent: List[Tuple[str, bool]]
    consequent: Tuple[str, bool]
    description: str
    severity: Severidade = Severidade.MEDIA
    action_prescribed: str = ""

    def evaluate_antecedent(self, working_memory: Dict[str, bool]) -> bool:
        """Avalia se todas as premissas conjuntivas (AND) são satisfeitas na memória de trabalho."""
        for fact_name, expected_val in self.antecedent:
            if fact_name not in working_memory:
                return False
            if working_memory[fact_name] != expected_val:
                return False
        return True

@dataclass
class AuditStep:
    step_number: int
    rule_fired: Optional[str]
    inferred_fact: str
    inferred_value: bool
    justification_facts: List[str]
    explanation: str

class InferenceEngine:
    """Motor Híbrido de Inferência Lógica para Sistemas Especialistas Industriais (SCADA-Core)."""
    def __init__(self, name: str = "SCADA-Core Inference Engine"):
        self.name = name
        self.knowledge_base: List[Rule] = []
        self.working_memory: Dict[str, bool] = {}
        self.audit_trail: List[AuditStep] = []

    def add_rule(self, rule: Rule) -> None:
        self.knowledge_base.append(rule)

    def load_telemetry_facts(self, telemetry_dict: Dict[str, bool]) -> None:
        self.working_memory = telemetry_dict.copy()
        self.audit_trail.clear()

    def set_fact(self, fact_name: str, value: bool) -> None:
        self.working_memory[fact_name] = value

    def run_forward_chaining(self, max_iterations: int = 50) -> Dict[str, bool]:
        """Executa Forward Chaining iterativamente até atingir o Ponto Fixo (Fixed Point)."""
        iteration = 0
        fired_rule_ids: Set[str] = set()

        while iteration < max_iterations:
            iteration += 1
            candidate_rules: List[Rule] = []
            for rule in self.knowledge_base:
                if rule.rule_id in fired_rule_ids:
                    continue
                if rule.evaluate_antecedent(self.working_memory):
                    conseq_name, conseq_val = rule.consequent
                    if conseq_name not in self.working_memory or self.working_memory[conseq_name] != conseq_val:
                        candidate_rules.append(rule)

            if not candidate_rules:
                break # Ponto Fixo atingido

            # Resolução de conflitos: Severidade (1 primeiro) > Especificidade (mais premissas)
            candidate_rules.sort(key=lambda r: (r.severity.value, -len(r.antecedent)))
            selected_rule = candidate_rules[0]

            conseq_name, conseq_val = selected_rule.consequent
            self.working_memory[conseq_name] = conseq_val
            fired_rule_ids.add(selected_rule.rule_id)

            premises_summary = [f"{k}={v}" for k, v in selected_rule.antecedent]
            step = AuditStep(
                step_number=len(self.audit_trail) + 1,
                rule_fired=selected_rule.rule_id,
                inferred_fact=conseq_name,
                inferred_value=conseq_val,
                justification_facts=premises_summary,
                explanation=f"Regra [{selected_rule.rule_id}] disparada: {selected_rule.description}. Ação: {selected_rule.action_prescribed}"
            )
            self.audit_trail.append(step)

        return self.working_memory

    def run_backward_chaining(self, goal_name: str, expected_value: bool = True) -> Tuple[bool, List[str]]:
        """Executa Backward Chaining (Goal-Driven) com busca em profundidade e prevenção de ciclos."""
        goal_stack: List[str] = []
        proof_trace: List[str] = []

        def prove_goal(target_fact: str, target_val: bool, depth: int = 0) -> bool:
            indent = "  " * depth
            goal_key = f"{target_fact}={target_val}"

            if target_fact in self.working_memory:
                actual_val = self.working_memory[target_fact]
                if actual_val == target_val:
                    proof_trace.append(f"{indent}[FATO COMPROVADO] {target_fact} = {target_val} (Presente na Memória)")
                    return True
                else:
                    proof_trace.append(f"{indent}[FATO CONTRADITÓRIO] {target_fact} = {actual_val} != {target_val}")
                    return False

            if goal_key in goal_stack:
                proof_trace.append(f"{indent}[CICLO DETECTADO] Meta {goal_key} já na pilha.")
                return False

            goal_stack.append(goal_key)
            proof_trace.append(f"{indent}[INVESTIGANDO META] Provar {target_fact} == {target_val}?")

            matching_rules = [
                r for r in self.knowledge_base
                if r.consequent[0] == target_fact and r.consequent[1] == target_val
            ]

            if not matching_rules:
                proof_trace.append(f"{indent}[FALHA] Nenhuma regra deriva {goal_key}")
                goal_stack.pop()
                return False

            for rule in matching_rules:
                proof_trace.append(f"{indent}--> Testando Regra [{rule.rule_id}]: {rule.description}")
                all_subgoals_proven = True

                for sub_fact, sub_val in rule.antecedent:
                    sub_proven = prove_goal(sub_fact, sub_val, depth + 1)
                    if not sub_proven:
                        all_subgoals_proven = False
                        break

                if all_subgoals_proven:
                    proof_trace.append(f"{indent}[META PROVADA VIA {rule.rule_id}] {target_fact} = {target_val}")
                    self.working_memory[target_fact] = target_val
                    goal_stack.pop()
                    return True

            goal_stack.pop()
            proof_trace.append(f"{indent}[FALHA NA META] {goal_key} não pôde ser sustentada.")
            return False

        success = prove_goal(goal_name, expected_value, 0)
        return success, proof_trace

    def explain_how(self, fact_name: str) -> List[str]:
        """Explica como um determinado fato foi inferido (HOW Explanation)."""
        steps = [s for s in self.audit_trail if s.inferred_fact == fact_name]
        if not steps:
            if fact_name in self.working_memory:
                return [f"O fato '{fact_name}' foi fornecido como entrada direta da telemetria de sensores."]
            return [f"O fato '{fact_name}' não foi estabelecido na memória de trabalho."]
        explanations = []
        for s in steps:
            explanations.append(
                f"Passo {s.step_number}: Derivado pela Regra [{s.rule_fired}] porque as premissas ({', '.join(s.justification_facts)}) foram satisfeitas. Detalhes: {s.explanation}"
            )
        return explanations

    def get_audit_trail_table(self) -> List[Dict[str, Any]]:
        """Retorna o Audit Trail em formato tabular para a IHM SCADA."""
        return [
            {
                "Passo": s.step_number,
                "Regra": s.rule_fired,
                "Fato Inferido": f"{s.inferred_fact} = {s.inferred_value}",
                "Premissas": ", ".join(s.justification_facts),
                "Explicação Operacional": s.explanation
            }
            for s in self.audit_trail
        ]

print("[OK] Motor Híbrido de Inferência Lógica inicializado com sucesso!")


[OK] Motor Híbrido de Inferência Lógica inicializado com sucesso!


## 1. Cadastro da Base de Conhecimento Oficial da Planta de Grãos

Definição das **Regras de Produção R01 a R12** e regras de intertravamento/trip da planta de seleção de grãos (Recepção, Esteira, Pesagem, Visão Computacional, Ejeção Pneumática e Silos).


In [2]:
def build_scada_core_knowledge_base() -> InferenceEngine:
    """Instancia o motor de inferência populado com as regras formais R01 a R12 e intertravamentos."""
    engine = InferenceEngine(name="SCADA-Core Automatica Expert Engine")

    # R01: Obstrução mecânica no funil
    engine.add_rule(Rule(
        rule_id="R01",
        antecedent=[("c_ALIM", True), ("p_MOV201", True), ("p_NB101", False), ("vazao_massa_nula", True)],
        consequent=("causa_obstrucao_funil", True),
        description="Obstrução mecânica na saída do funil de recepção",
        severity=Severidade.MEDIA,
        action_prescribed="Desligar alimentador e desobstruir grelha de passagem."
    ))

    # R02: Nível baixo no funil
    engine.add_rule(Rule(
        rule_id="R02",
        antecedent=[("p_NB101", True), ("c_ALIM", True)],
        consequent=("alerta_funil_vazio", True),
        description="Funil de recepção próximo ao desabastecimento",
        severity=Severidade.BAIXA,
        action_prescribed="Solicitar recarga imediata de matéria-prima."
    ))

    # R03: Travamento mecânico da esteira
    engine.add_rule(Rule(
        rule_id="R03",
        antecedent=[("c_ESTEIRA", True), ("p_JI201", True), ("p_MOV201", False)],
        consequent=("causa_travamento_esteira", True),
        description="Travamento mecânico no rolo de tração ou motor da esteira",
        severity=Severidade.CRITICA,
        action_prescribed="Bloqueio LOTO, inspeção mecânica de mancais e alívio de carga."
    ))

    # R04: Falha no encoder ST-201 ou patinagem
    engine.add_rule(Rule(
        rule_id="R04",
        antecedent=[("c_ESTEIRA", True), ("p_JI201", False), ("p_MOV201", False)],
        consequent=("causa_falha_encoder_st201", True),
        description="Falha de sinal no encoder ST-201 ou correia patinando no tambor",
        severity=Severidade.ALTA,
        action_prescribed="Inspecionar acoplamento do encoder incremental e esticador da correia."
    ))

    # R05: Sobrecarga na balança WT-301
    engine.add_rule(Rule(
        rule_id="R05",
        antecedent=[("sobrecarga_massa_wt301", True)],
        consequent=("causa_sobrecarga_pesagem", True),
        description="Sobrecarga excessiva de produto sobre a calha de pesagem",
        severity=Severidade.MEDIA,
        action_prescribed="Reduzir taxa vibratória do alimentador e checar célula de carga."
    ))

    # R06: Deriva de zero na balança WT-301
    engine.add_rule(Rule(
        rule_id="R06",
        antecedent=[("c_ALIM", False), ("p_MOV201", True), ("massa_residual_wt301", True)],
        consequent=("diagnostico_deriva_zero_wt301", True),
        description="Deriva de zero ou impregnação de pó na calha de pesagem",
        severity=Severidade.BAIXA,
        action_prescribed="Executar rotina de calibração de zero (tara) e limpeza da esteira."
    ))

    # R07: Falha na câmera KSA-401
    engine.add_rule(Rule(
        rule_id="R07",
        antecedent=[("p_KSA401", False)],
        consequent=("causa_falha_camera_visao", True),
        description="Falha de comunicação GigE ou travamento do software de visão",
        severity=Severidade.ALTA,
        action_prescribed="Reiniciar serviço de visão, checar link de rede e alimentação 24VDC."
    ))

    # R08: Lote contaminado
    engine.add_rule(Rule(
        rule_id="R08",
        antecedent=[("taxa_rejeicao_alta", True)],
        consequent=("diagnostico_lote_contaminado", True),
        description="Taxa de grãos categoria C acima do limite estatístico aceitável (> 35%)",
        severity=Severidade.MEDIA,
        action_prescribed="Emitir alerta ao controle de qualidade e segregar lote do produtor."
    ))

    # R09: Lente da câmera suja ou iluminação defeituosa
    engine.add_rule(Rule(
        rule_id="R09",
        antecedent=[("p_KSA401", True), ("p_XS401", True), ("rejeicao_anomala_consecutiva", True)],
        consequent=("causa_lente_suja_ou_luz", True),
        description="Lente da câmera obstruída por poeira ou módulo de iluminação LED queimado",
        severity=Severidade.ALTA,
        action_prescribed="Limpar vidro protetor da objetiva e testar luminária de alta frequência."
    ))

    # R10: Queda de pressão de ar comprimido (PAL-601)
    engine.add_rule(Rule(
        rule_id="R10",
        antecedent=[("p_PAL601", True)],
        consequent=("causa_queda_pressao_ar", True),
        description="Pressão na linha pneumática principal abaixo de 6 bar",
        severity=Severidade.ALTA,
        action_prescribed="Verificar compressor central, dreno de condensado e vazamentos na tubulação."
    ))

    # R11: Falha na solenoide ejetora FY-603
    engine.add_rule(Rule(
        rule_id="R11",
        antecedent=[("c_FY603", True), ("p_PAL601", False), ("p_ZSH601", False)],
        consequent=("causa_falha_solenoide_fy603", True),
        description="Válvula solenoide FY-603 não atuou fisicamente apesar do comando elétrico ativo",
        severity=Severidade.CRITICA,
        action_prescribed="Testar tensão 24V na bobina da solenoide e trocar válvula rápida."
    ))

    # R12: Silo de rejeito categoria C saturado
    engine.add_rule(Rule(
        rule_id="R12",
        antecedent=[("p_NC703", True)],
        consequent=("causa_silo_rejeito_cheio", True),
        description="Silo de descarte atingiu nível de 100% com risco iminente de transbordo",
        severity=Severidade.ALTA,
        action_prescribed="Substituir caçamba de rejeito e resetar permissivo na IHM."
    ))

    # Regras de Intertravamento Geral e Trip (Matriz de Causa e Efeito da Aula 07)
    engine.add_rule(Rule(
        rule_id="R_TRIP_EMERG",
        antecedent=[("p_EMERG", True)],
        consequent=("trip_geral", True),
        description="Trip de emergência por botão físico de soco acionado",
        severity=Severidade.CRITICA,
        action_prescribed="Desarme total de atuadores e travamento de segurança."
    ))

    engine.add_rule(Rule(
        rule_id="R_TRIP_PNEUM",
        antecedent=[("causa_queda_pressao_ar", True)],
        consequent=("trip_geral", True),
        description="Trip por perda do suprimento de pressão pneumática",
        severity=Severidade.CRITICA,
        action_prescribed="Inibir dosagem e ejeção para evitar contaminação de lote."
    ))

    engine.add_rule(Rule(
        rule_id="R_TRIP_SILO",
        antecedent=[("causa_silo_rejeito_cheio", True)],
        consequent=("trip_geral", True),
        description="Trip por sobreenchimento do silo de refugo",
        severity=Severidade.CRITICA,
        action_prescribed="Parar alimentação até descarte do reservatório."
    ))

    engine.add_rule(Rule(
        rule_id="R_DESLIGA_ALIM",
        antecedent=[("trip_geral", True)],
        consequent=("bloqueio_alimentador_c_ALIM", True),
        description="Bloqueio imediato do alimentador vibratório por Trip Geral",
        severity=Severidade.CRITICA,
        action_prescribed="Garantir c_ALIM = 0 para cessar entrada de grãos."
    ))

    return engine

engine_demo = build_scada_core_knowledge_base()
catalogo = [
    {
        "ID": r.rule_id,
        "Severidade": r.severity.name,
        "Premissas (SE)": " AND ".join([f"{k}={v}" for k, v in r.antecedent]),
        "Conclusão (ENTÃO)": f"{r.consequent[0]}={r.consequent[1]}",
        "Descrição": r.description
    }
    for r in engine_demo.knowledge_base
]

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (GRUPO 04) ===")
print(formatar_tabela(catalogo))


=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (GRUPO 04) ===
ID             | Severidade | Premissas (SE)                                                            | Conclusão (ENTÃO)                  | Descrição                                                                      
---------------+------------+---------------------------------------------------------------------------+------------------------------------+--------------------------------------------------------------------------------
R01            | MEDIA      | c_ALIM=True AND p_MOV201=True AND p_NB101=False AND vazao_massa_nula=True | causa_obstrucao_funil=True         | Obstrução mecânica na saída do funil de recepção                               
R02            | BAIXA      | p_NB101=True AND c_ALIM=True                                              | alerta_funil_vazio=True            | Funil de recepção próximo ao desabastecimento                                  
R03            | CRITICA    | c_ES

## 2. Cenário 1: Forward Chaining — Queda de Pressão Pneumática e Trip Geral em Cascata

**Condições Iniciais da Telemetria:**
- `p_PAL601 = True` (Pressostato indica $P < 6.0\text{ bar}$)
- `p_EMERG = False` (Botoeira de emergência livre)
- `p_JI201 = False` (Motor sem sobrecorrente)
- `p_KSA401 = True` (Câmera operacional)
- `p_MOV201 = True` (Esteira em movimento)
- `c_ALIM = True` (Alimentador ligado)
- `c_ESTEIRA = True` (Esteira ligada)

O motor de inferência deve deduzir em cascata:
$$\text{p\_PAL601} \xrightarrow{\text{R10}} \text{causa\_queda\_pressao\_ar} \xrightarrow{\text{R\_TRIP\_PNEUM}} \text{trip\_geral} \xrightarrow{\text{R\_DESLIGA\_ALIM}} \text{bloqueio\_alimentador\_c\_ALIM}$$


In [3]:
# Instanciação do motor
engine1 = build_scada_core_knowledge_base()

# Telemetria do Cenário 1
telemetria_cenario_1 = {
    "p_PAL601": True,
    "p_EMERG": False,
    "p_JI201": False,
    "p_KSA401": True,
    "p_MOV201": True,
    "p_NC703": False,
    "c_ALIM": True,
    "c_ESTEIRA": True,
}

engine1.load_telemetry_facts(telemetria_cenario_1)
memoria_saturada1 = engine1.run_forward_chaining()

print("Trilha de Diagnóstico Forward Chaining (Cenário 1):")
print(formatar_tabela(engine1.get_audit_trail_table()))

# Verificações de asserção
assert memoria_saturada1.get("causa_queda_pressao_ar") is True
assert memoria_saturada1.get("trip_geral") is True
assert memoria_saturada1.get("bloqueio_alimentador_c_ALIM") is True

print("\n--- Explicação Operacional HOW (bloqueio_alimentador_c_ALIM) ---")
for exp in engine1.explain_how("bloqueio_alimentador_c_ALIM"):
    print(exp)


Trilha de Diagnóstico Forward Chaining (Cenário 1):
Passo | Regra          | Fato Inferido                      | Premissas                   | Explicação Operacional                                                                                                                                          
------+----------------+------------------------------------+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------
1     | R10            | causa_queda_pressao_ar = True      | p_PAL601=True               | Regra [R10] disparada: Pressão na linha pneumática principal abaixo de 6 bar. Ação: Verificar compressor central, dreno de condensado e vazamentos na tubulação.
2     | R_TRIP_PNEUM   | trip_geral = True                  | causa_queda_pressao_ar=True | Regra [R_TRIP_PNEUM] disparada: Trip por perda do suprimento de pressão pneumática. Ação: Inibir 

## 3. Cenário 2: Forward Chaining — Travamento Mecânico do Rolo/Motor da Esteira

**Condições Iniciais da Telemetria:**
- `c_ESTEIRA = True` (Comando elétrico do motor ativo)
- `p_JI201 = True` (Relé térmico digital detecta sobrecorrente $I > 1.4 I_n$)
- `p_MOV201 = False` (Encoder ST-201 registra velocidade zero $\omega = 0$)


In [4]:
engine2 = build_scada_core_knowledge_base()
telemetria_cenario_2 = {
    "c_ESTEIRA": True,
    "p_JI201": True,
    "p_MOV201": False,
    "p_EMERG": False,
    "p_PAL601": False,
    "p_KSA401": True
}

engine2.load_telemetry_facts(telemetria_cenario_2)
memoria_saturada2 = engine2.run_forward_chaining()

print("Trilha de Diagnóstico Forward Chaining (Cenário 2):")
print(formatar_tabela(engine2.get_audit_trail_table()))

assert memoria_saturada2.get("causa_travamento_esteira") is True
print("\n[OK] Diagnóstico Crítico de Travamento Mecânico inferido com sucesso!")


Trilha de Diagnóstico Forward Chaining (Cenário 2):
Passo | Regra | Fato Inferido                   | Premissas                                    | Explicação Operacional                                                                                                                                
------+-------+---------------------------------+----------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------
1     | R03   | causa_travamento_esteira = True | c_ESTEIRA=True, p_JI201=True, p_MOV201=False | Regra [R03] disparada: Travamento mecânico no rolo de tração ou motor da esteira. Ação: Bloqueio LOTO, inspeção mecânica de mancais e alívio de carga.

[OK] Diagnóstico Crítico de Travamento Mecânico inferido com sucesso!


## 4. Cenário 3: Backward Chaining — Investigação de Falha na Válvula Ejetora FY-603

**Problema Operacional:** O operador do SCADA observa que grãos de Categoria C não estão sendo ejetados adequadamente e submete a meta de investigação à IHM:
$$\text{Meta: } \text{causa\_falha\_solenoide\_fy603} \stackrel{?}{=} \text{True}$$

**Fatos Ativos na Planta:**
- `c_FY603 = True` (Pulso elétrico de acionamento enviado pelo CLP)
- `p_PAL601 = False` (Pressão pneumática nominal em 7.0 bar)
- `p_ZSH601 = False` (Sensor magnético de fim de curso NÃO confirmou avanço físico do êmbolo)


In [5]:
engine3 = build_scada_core_knowledge_base()
engine3.load_telemetry_facts({
    "c_FY603": True,
    "p_PAL601": False,
    "p_ZSH601": False,
})

sucesso3, trace3 = engine3.run_backward_chaining("causa_falha_solenoide_fy603", True)

print(f"Meta 'causa_falha_solenoide_fy603' Provada? {sucesso3}\n")
print("Árvore de Prova (SLD Resolution Trace):")
for step in trace3:
    print(step)

assert sucesso3 is True
assert engine3.working_memory.get("causa_falha_solenoide_fy603") is True


Meta 'causa_falha_solenoide_fy603' Provada? True

Árvore de Prova (SLD Resolution Trace):
[INVESTIGANDO META] Provar causa_falha_solenoide_fy603 == True?
--> Testando Regra [R11]: Válvula solenoide FY-603 não atuou fisicamente apesar do comando elétrico ativo
  [FATO COMPROVADO] c_FY603 = True (Presente na Memória)
  [FATO COMPROVADO] p_PAL601 = False (Presente na Memória)
  [FATO COMPROVADO] p_ZSH601 = False (Presente na Memória)
[META PROVADA VIA R11] causa_falha_solenoide_fy603 = True


## 5. Cenário 4: Backward Chaining — Investigação de Hipótese Refutada (Lente Suja vs Normal)

**Hipótese Sob Teste:**
$$\text{Meta: } \text{causa\_lente\_suja\_ou\_luz} \stackrel{?}{=} \text{True}$$

**Fatos Ativos:**
- `p_KSA401 = True` (Câmera online)
- `p_XS401 = True` (Presença de grão detectada no foco óptico)
- `rejeicao_anomala_consecutiva = False` (Grãos classificados normalmente sem falso-rejeito massivo)


In [6]:
engine4 = build_scada_core_knowledge_base()
engine4.load_telemetry_facts({
    "p_KSA401": True,
    "p_XS401": True,
    "rejeicao_anomala_consecutiva": False,
})

sucesso4, trace4 = engine4.run_backward_chaining("causa_lente_suja_ou_luz", True)

print(f"Meta 'causa_lente_suja_ou_luz' Provada? {sucesso4}\n")
print("Árvore de Prova (SLD Resolution Trace):")
for step in trace4:
    print(step)

assert sucesso4 is False
print("\n[OK] Hipótese refutada com sucesso devido a contradição com os fatos da planta!")


Meta 'causa_lente_suja_ou_luz' Provada? False

Árvore de Prova (SLD Resolution Trace):
[INVESTIGANDO META] Provar causa_lente_suja_ou_luz == True?
--> Testando Regra [R09]: Lente da câmera obstruída por poeira ou módulo de iluminação LED queimado
  [FATO COMPROVADO] p_KSA401 = True (Presente na Memória)
  [FATO COMPROVADO] p_XS401 = True (Presente na Memória)
  [FATO CONTRADITÓRIO] rejeicao_anomala_consecutiva = False != True
[FALHA NA META] causa_lente_suja_ou_luz=True não pôde ser sustentada.

[OK] Hipótese refutada com sucesso devido a contradição com os fatos da planta!


## 6. Cenário 5: Diagnóstico Integrado Multivariável e Relatório de Auditoria (SOE)

Simulação de múltiplos eventos simultâneos: Silo de rejeito saturado (`p_NC703 = True`) em conjunto com acúmulo de pó na balança WT-301 (`massa_residual_wt301 = True`, `c_ALIM = False`, `p_MOV201 = True`).


In [7]:
engine5 = build_scada_core_knowledge_base()
telemetria_cenario_5 = {
    "p_NC703": True,
    "massa_residual_wt301": True,
    "c_ALIM": False,
    "p_MOV201": True,
    "p_EMERG": False,
    "p_JI201": False,
    "p_PAL601": False,
    "p_KSA401": True
}

engine5.load_telemetry_facts(telemetria_cenario_5)
memoria_saturada5 = engine5.run_forward_chaining()

print("=== RELATÓRIO DE AUDITORIA FORMAL (AUDIT TRAIL / SOE) ===")
print(formatar_tabela(engine5.get_audit_trail_table()))

assert memoria_saturada5.get("causa_silo_rejeito_cheio") is True
assert memoria_saturada5.get("diagnostico_deriva_zero_wt301") is True
assert memoria_saturada5.get("trip_geral") is True
assert memoria_saturada5.get("bloqueio_alimentador_c_ALIM") is True

print("\n[SUCESSO] Todos os testes de inferência e validação do SCADA-Core foram concluídos com êxito!")


=== RELATÓRIO DE AUDITORIA FORMAL (AUDIT TRAIL / SOE) ===
Passo | Regra          | Fato Inferido                        | Premissas                                              | Explicação Operacional                                                                                                                                          
------+----------------+--------------------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------
1     | R12            | causa_silo_rejeito_cheio = True      | p_NC703=True                                           | Regra [R12] disparada: Silo de descarte atingiu nível de 100% com risco iminente de transbordo. Ação: Substituir caçamba de rejeito e resetar permissivo na IHM.
2     | R_TRIP_SILO    | trip_geral = True                    | causa_silo_rejeito_cheio=True   